# AI Sommelier RAG: 이미지와 와인 리뷰로 추천하기

이 노트북은 검색 없는 추천과 RAG 추천을 비교한다. 요리 이미지를 검색 query로 바꾸고, 앞 노트북에 저장한 Wine Magazine 리뷰를 근거로 와인을 추천한다.

### 전체 흐름

`이미지 URL → 풍미 query → Retriever → list[Document] → context → 와인 추천`

### Baseline과 RAG의 차이

- Baseline: 모델의 사전학습 지식만 사용한다.
- RAG: Pinecone에서 찾은 실제 와인 리뷰를 Prompt의 근거로 사용한다.

### 사용하는 입력과 저장소

- Multimodal message: 텍스트와 이미지 URL을 한 메시지에 담는다.
- Query embedding: 이미지에서 만든 풍미 문장을 숫자 vector로 바꾼다.
- Pinecone index: 앞 노트북에서 만든 `winemag-review-data`를 사용한다.
- 기본 namespace: 별도 namespace를 지정하지 않은 레코드를 검색한다.

RAG의 인덱싱은 앞 노트북에서 완료했다. 여기서는 요청마다 Retrieval과 Generation을 실행한다. 이미지 해석은 Retrieval에 넣을 query를 만드는 전처리 단계이다.


## 패키지 준비

이 실습은 LangChain Runnable로 OpenAI 모델과 Pinecone Retriever를 연결한다.

- `langchain`: Prompt, Parser, Runnable을 제공한다.
- `langchain-openai`: 이미지 Chat Model과 query embedding을 제공한다.
- `langchain-pinecone`: Pinecone index를 LangChain Retriever로 연결한다.
- `pinecone`: Pinecone 서비스에 접속하는 공식 SDK이다.
- `python-dotenv`: `.env`의 API 설정을 환경 변수로 불러온다.
- `langchain-community`: 앞 인덱싱 단계의 `CSVLoader`와 같은 실습 환경을 유지한다.


In [1]:
from langchain_community.tools.azure_ai_services import image_analysis
%pip install -U langchain langchain-openai langchain-pinecone langchain-community pinecone python-dotenv


  Using cached pinecone-9.1.0-cp310-abi3-win_amd64.whl.metadata (6.3 kB)
Note: you may need to restart the kernel to use updated packages.


## API 설정 불러오기

`.env`의 API 키는 각 SDK의 인증에 사용하며 화면에 출력하지 않는다.

### 필요한 설정

- `OPENAI_API_KEY`: 이미지 해석, embedding, 추천 생성에 사용한다.
- `PINECONE_API_KEY`: `winemag-review-data` index를 검색할 때 사용한다.
- `OPENAI_EMBEDDING_MODEL`: 앞 인덱싱 단계에서 `text-embedding-3-small`로 설정한다.

### 앞 노트북과 같아야 하는 값

- embedding 모델: `text-embedding-3-small`이다.
- index 이름: `winemag-review-data`이다.
- namespace: 별도 값을 지정하지 않은 기본 namespace이다.

Pinecone index는 앞 인덱싱 노트북에서 생성·적재가 끝난 상태여야 한다.


In [2]:
import os

from dotenv import load_dotenv

load_dotenv()

CHAT_MODEL_NAME = os.getenv('OPENAI_CHAT_MODEL', 'gpt-5.6-luna')


## Baseline 1: 요리 이름으로 와인 추천하기

**Baseline**은 개선 방법의 효과를 비교하기 위한 기준 실행이다. 첫 번째 baseline은 외부 문서를 검색하지 않는다.

- 입력: 요리 이름 `도미회`이다.
- 처리: Prompt와 모델의 사전학습 지식만 사용한다.
- 출력: 와인 추천 문자열이다.
- 한계: 추천에 사용한 외부 리뷰와 출처가 없다.

LCEL의 `|`는 Runnable을 왼쪽에서 오른쪽으로 연결한다. 앞 단계의 출력이 다음 단계의 입력이 된다.


### Text Prompt → Chat Model → 문자열

세 Runnable을 `|`로 연결한다.

- `ChatPromptTemplate`: `system` 지시와 `{query}`를 역할별 message로 만든다.
- `ChatOpenAI`: message를 받아 `AIMessage`를 생성한다.
- `StrOutputParser`: `AIMessage`에서 답변 텍스트만 꺼내 `str`로 바꾼다.

입력과 출력은 `{'query': str} → PromptValue → AIMessage → str` 순서로 변한다.


In [6]:
from langchain_core.prompts import ChatPromptTemplate

from langchain_core.output_parsers import StrOutputParser

from langchain_openai import ChatOpenAI

# 1. Prompt 생성
text_recommendation_prompt = ChatPromptTemplate.from_messages([
    ('system', '''페르소나: 당신은 와인과 음식 페어링에 대한 열정을 지닌 지식 많고 경험 풍부한 소믈리에이다. 다양한 와인 산지, 포도 품종, 테이스팅 노트에 대한 깊은 이해를 갖고 있다. 친근하고 다가가기 쉬운 태도로 초보자부터 전문가까지 모두가 와인을 즐길 수 있도록 돕는다.

역할: 소믈리에로서 각종 요리에 완벽하게 어울리는 와인을 전문가 수준으로 추천한다. 이용자가 새로운 와인을 탐험하도록 안내하며 와인 테이스팅의 섬세함을 이해할 수 있게 지원한다. 적절한 와인을 매치해 식사 경험을 한층 더 풍성하게 만드는 것이 목표이다.

예시:

* 구운 마늘 버터 새우를 위해서는 샤르도네나 알바리뇨를 추천한다. 와인의 산도가 버터의 풍미와 기름진 맛을 깔끔하게 잡아준다.
* 가성비 좋은 와인을 찾는다면 프랑스 남부의 뮈스카데나나 스페인 리베라 델 두에로 지역의 템프라니요를 추천한다. 각각의 풍미 프로필과 어울리는 요리를 함께 설명한다.
* 와인 보관 방법을 묻는다면 적정 온도(12–14℃), 습도(60–70%), 빛 차단과 진동 방지 등의 실용적인 팁을 제공한다.'''),
    ('human', '''
다음 요리에 어울리는 와인을 추천해주세요. (한국말로 답변해주세요.)

요리명: {query}
'''),
])


# 2. baseline_llm 생성
baseline_llm = ChatOpenAI(
    model_name = CHAT_MODEL_NAME,
    temperature = 1, # 답변 다영성 허용
    use_responses_api=True,
    reasoning_effort='none' # 추가 추론 허용 X
    # (none, low, medium, high)
)

# 3. AIMessage -> str으로 변환하는 객체
output_parser = StrOutputParser()

# 4. 3개의 Runnable을 연결(chain)
text_baseline_chain = (
    text_recommendation_prompt
    | baseline_llm
    | output_parser
)

# 5. baseline 결과 확인
print(text_baseline_chain.invoke({'query':'도미회'}))


도미회에는 **산도가 선명하고 향이 과하지 않으며, 깔끔한 미네랄감이 있는 화이트 와인**이 잘 어울립니다. 도미의 섬세한 단맛과 탱글한 식감을 살리면서 비린 맛을 정리해주기 때문입니다.

### 가장 추천하는 와인

**1. 알바리뇨(Albariño)**
- 스페인 갈리시아의 대표적인 해산물용 와인
- 레몬, 자몽, 흰 꽃 향과 짭조름한 미네랄감
- 도미회의 담백함과 바다 향을 자연스럽게 연결해줍니다.
- 간장, 와사비와 곁들여도 산도가 균형을 잡아줍니다.

**2. 뮈스카데(Muscadet)**
- 프랑스 루아르 지역의 드라이 화이트
- 매우 깔끔하고 가벼우며, 은은한 효모감과 해조류 같은 미네랄 풍미
- 회 자체의 신선함을 가장 방해하지 않는 스타일입니다.

**3. 소비뇽 블랑(Sauvignon Blanc)**
- 뉴질랜드산은 상큼한 허브와 시트러스 풍미가 뚜렷합니다.
- 도미회에 레몬이나 유자, 초고추장을 곁들일 때 특히 좋습니다.
- 다만 향이 너무 강한 제품보다는 드라이하고 절제된 스타일을 고르세요.

### 부드러운 스타일을 원한다면

**샤블리(Chablis)**
- 오크 숙성이 강하지 않은 샤르도네
- 풋사과, 레몬, 백악질 미네랄 풍미가 있으며 질감이 매끄럽습니다.
- 도미회의 담백한 맛을 해치지 않으면서 조금 더 고급스럽고 둥근 인상을 줍니다.

### 한국식 곁들임별 추천

- **간장·와사비:** 알바리뇨, 샤블리  
- **초고추장:** 소비뇽 블랑, 드라이 리슬링  
- **참기름이나 깻잎을 곁들인 회:** 향이 너무 강하지 않은 알바리뇨 또는 드라이 리슬링  
- **소주 대신 가볍게 즐길 때:** 차갑게 마시는 프로세코 브뤼  

### 서빙 팁
화이트 와인은 **8–10℃** 정도로 차갑게 준비하고, 너무 향이 강하거나 오크 풍미가 진한 샤르도네는 피하는 것이 좋습니다. 한 병만 고른다면 **알바리뇨**를 가장 먼저 추천드립니다.


## Baseline 2: 요리 이미지로 와인 추천하기

**Multimodal message**는 한 메시지에 텍스트와 이미지처럼 서로 다른 입력 형식을 함께 담는다.

- `text` block: 모델이 수행할 요청을 전달한다.
- `image_url` block: 모델이 읽을 공개 이미지 주소를 전달한다.
- 출력: 이미지에서 추정한 요리와 와인 추천 문자열이다.

이미지 URL은 모델 서버가 접근할 수 있어야 한다. 이 단계도 Wine Magazine 리뷰는 검색하지 않는다.


In [7]:
from langchain_core.prompts import HumanMessagePromptTemplate

# 1. Prompt 생성
# - HumanMessagePromptTemplate을 이용해서 MultiModal 입력 만들기
image_recommendation_prompt = ChatPromptTemplate.from_messages([
    ('system', '''페르소나: 당신은 와인과 음식 페어링에 대한 열정을 지닌 지식 많고 경험 풍부한 소믈리에이다. 다양한 와인 산지, 포도 품종, 테이스팅 노트에 대한 깊은 이해를 갖고 있다. 친근하고 다가가기 쉬운 태도로 초보자부터 전문가까지 모두가 와인을 즐길 수 있도록 돕는다.

역할: 소믈리에로서 각종 요리에 완벽하게 어울리는 와인을 전문가 수준으로 추천한다. 이용자가 새로운 와인을 탐험하도록 안내하며 와인 테이스팅의 섬세함을 이해할 수 있게 지원한다. 적절한 와인을 매치해 식사 경험을 한층 더 풍성하게 만드는 것이 목표이다.

예시:

* 구운 마늘 버터 새우를 위해서는 샤르도네나 알바리뇨를 추천한다. 와인의 산도가 버터의 풍미와 기름진 맛을 깔끔하게 잡아준다.
* 가성비 좋은 와인을 찾는다면 프랑스 남부의 뮈스카데나나 스페인 리베라 델 두에로 지역의 템프라니요를 추천한다. 각각의 풍미 프로필과 어울리는 요리를 함께 설명한다.
* 와인 보관 방법을 묻는다면 적정 온도(12–14℃), 습도(60–70%), 빛 차단과 진동 방지 등의 실용적인 팁을 제공한다.'''),
    HumanMessagePromptTemplate.from_template([
        {'text': '다음 요리에 어울리는 와인을 추천해주세요. (한국말로 답변해주세요.)'},
        {'image_url': '{image_url}'},
    ]),
])

# 2. chain 구성
image_baseline_chain = (
    image_recommendation_prompt
    | baseline_llm
    | output_parser
)

# 3. chain.invoke()
# 이때, '{image_url}' 자리에 들어갈 이미지 주소 작성
image_url = 'https://ecimg.cafe24img.com/pg1819b68327228073/hyundeok2025/web/product/big/20250424/a1f31ef3fe068b5b28829425523d1212.jpg'

print(image_baseline_chain.invoke({'image_url':image_url}))

사진 속 요리는 **구운 돼지고기와 김치, 매콤한 콩나물무침·파채, 버섯과 양파**가 곁들여진 구성으로 보입니다. 돼지고기의 지방과 구운 풍미, 김치의 매운맛과 산미를 함께 고려하면 **산도가 좋고 약간 차갑게 마시는 와인**이 가장 잘 어울립니다.

### 가장 추천: 독일 리슬링 카비넷 또는 드라이 리슬링
- **향과 맛:** 라임, 풋사과, 복숭아, 상쾌한 산미
- **페어링 이유:** 높은 산도가 삼겹살의 기름기를 깔끔하게 씻어주고, 약간의 잔당이 김치와 고춧가루의 매운맛을 부드럽게 감싸줍니다.
- 매운 반찬이 많다면 완전히 드라이한 리슬링보다 **살짝 단맛이 있는 카비넷**이 특히 좋습니다.
- **추천 온도:** 8–10℃

### 가볍고 상쾌한 선택: 스페인 알바리뇨
- 레몬, 자몽, 사과 같은 산뜻한 풍미와 짭짤한 미네랄감이 있습니다.
- 김치와 파채의 새콤매콤한 맛을 살리면서 돼지고기의 느끼함을 정리해 줍니다.
- 단맛이 거의 없어 깔끔한 스타일을 선호할 때 좋습니다.

### 레드 와인을 원한다면: 보졸레 크뤼 또는 피노 누아
#### 보졸레 빌라주·크뤼
- 체리, 라즈베리, 은은한 꽃 향과 낮은 타닌이 특징입니다.
- 돼지고기와 잘 어울리면서도 김치의 매운맛을 지나치게 자극하지 않습니다.
- **약간 차갑게 12–14℃**로 마시면 더욱 좋습니다.

#### 뉴질랜드 피노 누아
- 붉은 과실, 허브, 부드러운 향신료 풍미가 구운 고기와 잘 어울립니다.
- 다만 오크 향이 강하고 타닌이 묵직한 피노 누아는 김치의 매운맛과 충돌할 수 있으므로, **가볍고 산뜻한 스타일**을 고르세요.

### 가성비 좋은 선택
- **포르투갈 비뉴 베르드:** 가볍고 산뜻하며 약한 탄산감이 있어 기름진 돼지고기에 좋습니다.
- **이탈리아 람브루스코 세코:** 드라이한 붉은 스파클링으로, 돼지고기와 김치 모두에 잘 어울립니다.
- **칠레 소비뇽 블랑:** 허브와 시트러스 풍미가 파채·콩나물무침과 잘 맞습니다.

**한 병만 고른다면:**  
매운맛까지 고려해

## Baseline 3: 와인 이미지로 요리 추천하기

세 번째 baseline은 입력과 출력의 방향을 바꾼다.

- 입력: 와인 라벨 이미지 URL이다.
- 처리: 라벨과 와인 특징을 multimodal 모델이 해석한다.
- 출력: 어울리는 요리 추천 문자열이다.

같은 Prompt → Model → Parser 구조에서도 system 역할과 human 질문을 바꾸면 Chain의 목적이 달라진다.


In [8]:
wine_image_prompt = ChatPromptTemplate.from_messages([
    ('system', '''페르소나(Persona): 당신은 와인과 음식의 조화를 깊이 이해하는 경험 많은 소믈리에이다. 다양한 와인 산지, 포도 품종, 테이스팅 노트에 대해 해박하며, 누구에게나 친근하고 쉽게 설명하는 능력을 가지고 있다.

역할(Role): 소믈리에로서, 당신의 역할은 특정 와인에 가장 잘 어울리는 요리를 전문적으로 추천하는 것이다. 와인의 향, 맛, 산도, 바디감 등을 분석해 최적의 음식 조합을 제안한다. 당신의 목표는 손님이 가진 와인을 더욱 특별하게 즐길 수 있도록, 완벽한 음식 페어링을 안내하는 것이다.

예시(Examples):

- 누군가가 ‘리슬링(Riesling)’ 와인을 가지고 있다고 하면, 와인의 상큼한 산도와 과일향에 어울리는 매콤한 아시아 요리나 스파이시 치킨을 추천하고 그 이유를 설명한다.
- ‘까베르네 소비뇽(Cabernet Sauvignon)’ 와인에 맞는 음식을 물어보면, 풍부한 탄닌과 바디감을 살려줄 스테이크나 구운 양고기와의 페어링을 안내한다.
- ‘스파클링 와인’에 잘 어울리는 간단한 핑거푸드나 해산물 요리 등을 추천하며, 와인의 청량감을 살리는 방법을 알려준다.
- 특정 와인을 활용한 요리 레시피를 제안하거나, 와인과 함께 먹으면 맛의 밸런스가 좋아지는 음식 스타일을 설명한다.'''),
    HumanMessagePromptTemplate.from_template([
        {'text': '다음 와인에 어울리는 요리를 추천해주세요. (한국말로 답변해주세요.)'},
        {'image_url': '{image_url}'},
    ]),
])

image_baseline_chain = (
    image_recommendation_prompt
    | baseline_llm
    | output_parser
)

image_url = 'https://blog.lgchem.com/wp-content/uploads/2015/10/1350761668_79bf0202_2014122902479_010.jpg'

print(image_baseline_chain.invoke({'image_url':image_url}))

사진 속 와인은 **카시에로 델 디아블로 리저브 카베르네 소비뇽(Casillero del Diablo Reserva Cabernet Sauvignon)**입니다. 칠레산 카베르네 소비뇽으로, 검은 과실·블랙커런트·자두 향과 은은한 오크, 적당한 탄닌과 산미가 특징입니다.

### 잘 어울리는 요리
- **소고기 스테이크, 등심·채끝 구이**  
  와인의 탄닌이 고기의 지방과 만나 부드러워지고, 풍부한 과실 향이 구운 풍미를 살려줍니다.
- **숯불갈비·소갈비찜**  
  달짝지근한 양념과 와인의 검은 과실 풍미가 잘 맞습니다. 단, 양념이 너무 맵거나 달면 와인이 다소 떫게 느껴질 수 있습니다.
- **불고기**  
  간장 베이스의 감칠맛과 카베르네 소비뇽의 오크 풍미가 조화를 이룹니다.
- **햄버거·미트볼·라구 파스타**  
  토마토와 육류의 진한 맛을 와인의 산미와 탄닌이 균형 있게 받쳐줍니다.
- **버섯구이 또는 렌틸·콩을 곁들인 요리**  
  버섯의 흙내음과 와인의 오크·허브 뉘앙스가 자연스럽게 이어집니다.

### 추천하지 않는 조합
회, 굴 같은 섬세한 해산물이나 아주 매운 음식과는 탄닌과 알코올감이 도드라질 수 있어 피하는 편이 좋습니다.

### 서빙 팁
**16–18℃**로 살짝 서늘하게 서빙하고, 개봉 후 **15~30분 정도** 두면 향이 더 부드럽게 펼쳐집니다.  
가장 추천하는 조합은 **소금과 후추만 가볍게 한 채끝 스테이크**입니다.


## Wine Magazine 기반 2-step RAG

RAG 본체는 Retrieval과 Generation 두 단계로 구성된다. 이미지 해석은 검색용 query를 만드는 전처리이다.

### 처리 순서

1. 이미지 해석: 이미지 URL을 풍미 query 문자열로 바꾼다.
2. Retrieval: query와 가까운 리뷰를 `list[Document]`로 가져온다.
3. Context 구성: 검색된 `Document.page_content`를 문자열 하나로 묶는다.
4. Generation: 풍미와 context를 사용해 와인을 추천한다.

`이미지 URL → 풍미 str → list[Document] → context str → 추천 str`


### LCEL에서 사용할 Runnable

`Runnable`은 입력을 받아 한 작업을 수행하고 출력을 반환하는 공통 실행 단위이다. `|`로 여러 Runnable을 연결하면 `RunnableSequence`가 만들어진다.

- `RunnableLambda`: 일반 Python 함수를 LCEL 단계로 사용한다.
- `RunnableParallel`: 같은 입력을 여러 분기에 전달하고 결과를 dict로 묶는다.
- `RunnablePassthrough`: 입력값을 바꾸지 않고 그대로 반환한다.

Retrieval 단계는 같은 풍미 문자열을 두 곳에서 사용한다.

- 보존 분기: `dish_flavor`에 원래 문자열을 남긴다.
- 검색 분기: Retriever가 `retrieved_documents`를 만든다.
- 병합 결과: `{'dish_flavor': str, 'retrieved_documents': list[Document]}`이다.


### 이미지 URL을 검색 query로 바꾸기

Pinecone에는 텍스트 와인 리뷰가 저장되어 있다. 따라서 요리 이미지를 바로 비교하지 않고 텍스트 풍미로 변환한다.

- 입력: `{'image_urls': list[str]}`이다.
- 변환: 각 URL을 multimodal image block으로 만든다.
- 출력: 검색에 사용할 영어 한 문장이다.
- 다음 사용처: Pinecone Retriever의 query로 전달한다.

`describe_dish_flavor()`는 결과 문자열을 직접 반환하지 않는다. 이미지 Prompt → Model → Parser를 연결한 Runnable을 반환한다.


In [9]:
def describe_dish_flavor(query: dict):

    #1. 음식 이미지 -> 영어 한 문장으로 변환 요청 Prompt 생성
    dish_flavor_prompt = ChatPromptTemplate.from_messages([
            ('system', '''페르소나: 당신은 조리 기법, 풍미 특성과 식재료 조합을 깊이 이해하는 뛰어난 음식 전문가이다. 다양한 요리를 탐구하는 데 열정이 있으며 음식의 감각적 경험을 구체적으로 표현할 수 있다. 실무 경험과 이론 지식을 모두 갖추고 있어 신뢰할 수 있는 분석을 제공한다.

    역할: 음식 전문가로서 다양한 요리의 맛, 식감과 향을 분석한다. 식재료와 조리 방법을 구체적으로 평가하고, 균형 있고 조화로운 요리를 만드는 원리를 설명한다. 또한 요리 기술을 향상하고 미식의 가치를 이해할 수 있도록 돕는다.

    예시:

    요리의 풍미 특성을 분석할 때는 산미, 단맛, 쓴맛과 감칠맛의 균형을 설명하고 이 요소들이 어떻게 어우러져 복합적인 맛을 만드는지 분석한다.
    특정 식재료의 풍미를 살리는 방법을 묻는다면 양파를 캐러멜화해 깊은 맛을 내거나 고기의 본연의 맛을 살리도록 적절히 간하는 방법처럼 실용적인 조언을 제공한다.
    음식 조합을 설명할 때는 해산물에 감귤류를 곁들여 산뜻함을 더하거나 허브로 전체 풍미를 끌어올리는 사례처럼 서로 보완하는 식재료와 풍미를 제안하고 그 이유를 설명한다.'''),
            ('human', '''
    이미지를 바탕으로 요리를 분석한다. 와인 리뷰 검색 query로 사용할 수 있도록 핵심 재료, 조리법과 풍미를 영어 한 문장으로 간결하게 출력한다.
    '''),
        ])

    # 2. image_url의 각 문자열을 multimodal image content block으로 변경
    image_contents = [
        {'image_url': image_url}
        for image_url in query.get('image_urls', [])
    ]

    # 3. HumanMessage로 변환하여 Prompt에 추가
    dish_flavor_prompt += HumanMessagePromptTemplate.from_template(image_contents)

    # 4. 이미지를 영어 한 문장으로 변환할 llm 준비
    image_analysis_llm = ChatOpenAI(
        model_name=CHAT_MODEL_NAME,
        use_responses_api=True,
        temperature=0,
        reasoning_effort='none'
    )

    # 5. chain 구성 후 반환
    return dish_flavor_prompt | image_analysis_llm | StrOutputParser()



### 이미지 해석 Runnable 실행하기

`RunnableLambda`는 일반 Python 함수를 `invoke()`로 실행할 수 있게 감싼다.

이 예제에는 한 단계가 더 있다.

1. `RunnableLambda`가 `describe_dish_flavor(payload)`를 호출한다.
2. 함수가 Prompt → Model → Parser Runnable을 반환한다.
3. LangChain이 반환된 Runnable에도 같은 `payload`를 전달해 이어서 실행한다.
4. 최종 결과로 풍미 문자열을 반환한다.

출력은 음식·조리법·풍미가 포함된 검색용 영어 한 문장이어야 한다.


In [10]:
from langchain_core.runnables import RunnableLambda

# 1. 일반 함수 describe_dish_flavor를
# LCEL에서 실행할 수 있도록 RunnableLambda로 감싸기
describe_dish_flavor_chain = RunnableLambda(describe_dish_flavor)

# 2. 이미지 주소를 전달하여 결과 확인
dish_flavor = describe_dish_flavor_chain.invoke({
    'image_urls': [
        'https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTUYO3qbwc9-sjFx_NlzybanjZ1T4I8ynfWhHgHOQc7ow&s'
    ]
})

print(dish_flavor)

Korean-style spicy stir-fried noodles with pork, onions, and cabbage, featuring chewy noodles coated in a savory, sweet, spicy, and umami-rich sauce.


### Pinecone index를 Retriever로 연결하기

앞 노트북은 와인 리뷰를 `text-embedding-3-small`로 변환해 `winemag-review-data` index의 기본 namespace에 저장했다.

### 검색 설정

- embedding 모델: 문서 인덱싱과 같은 `text-embedding-3-small`이다.
- index: `winemag-review-data`이다.
- namespace: 별도 값을 지정하지 않은 기본 namespace이다.
- 검색 방식: 의미 유사도 검색이다.
- `k=5`: 상위 리뷰를 최대 다섯 개 반환한다.

`as_retriever()`는 Vector Store를 `query str → list[Document]` 구조의 검색 인터페이스로 바꾼다.


In [12]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

# 1. 임베딩 객체 준비
embeddings = OpenAIEmbeddings(model=os.environ['OPENAI_EMBEDDING_MODEL'])

# 2. Pinecone Index와 임베딩 객체를 Vector Store로 묶기
wine_vector_store = PineconeVectorStore(
    index_name = 'winemag-review-data',
    embedding=embeddings,
)

# 3. Vector Store를 query:str -> list[Document] 구조로 반환할
# Retriever로 변경하기
wine_retriever = wine_vector_store.as_retriever(
    search_type='similarity',
    search_kwargs={'k':5}
)


### 대표 query로 검색 결과 확인하기

이미지 해석 단계와 같은 형식의 영어 풍미 문장을 Retriever에 전달한다.

- 입력: 음식의 재료·조리법·풍미를 설명한 문자열이다.
- 출력: 유사도 순서의 `list[Document]`이다.
- 확인할 값: 각 문서의 `page_content`와 `metadata`이다.

검색 결과가 존재한다고 항상 관련성이 높은 것은 아니다. 생성 전에 음식 풍미와 실제 와인 리뷰가 연결되는지 읽어 본다.


In [13]:
sample_dish_flavor = (
    'A succulent roast beef garnished with fresh rosemary and accompanied '
    'by vibrant cherry tomatoes and roasted vegetables.'
)

# query -> Retriever -> list[Document]
retrieved_documents = wine_retriever.invoke(sample_dish_flavor)

for rank, document in enumerate(retrieved_documents, start=1):
    print(f'[{rank}] metadata:', document.metadata)
    print(document.page_content)
    print()


[1] metadata: {'row': 28592.0, 'source': './winemag-data-130k-v2.csv'}
: 28592
country: US
description: Cherry, pepper and warming spices lead on the red from Virginia. On the palate, cedar, spice and pepper are delivered in a big package. Pair with grilled meat, spicy cuisine.
designation: Rooster Red Wine
points: 83
price: 20.0
province: Virginia
region_1: Virginia
region_2: 
taster_name: Susan Kostrzewa
taster_twitter_handle: @suskostrzewa
title: Veramar NV Rooster Red Wine Red (Virginia)
variety: Bordeaux-style Red Blend
winery: Veramar

[2] metadata: {'row': 12969.0, 'source': './winemag-data-130k-v2.csv'}
: 12969
country: US
description: Roasted and raw beef, dried strawberry, crushed marjoram and salt-and-pepper seasonings show on this Syrah. Extremely dark in the glass, it's bolstered by 4% Grenache, 4% Mourvèdre and 2% Viognier. Elderberry fruit kicks off the palate, following by pepper, coffee grinds, espresso beans and dark-chocolate sauce, surrounded by grippy tannins.
desi

### 검색 결과를 Generation 입력으로 바꾸기

Generation Prompt에는 원래 풍미와 검색 리뷰가 모두 필요하다.

### 두 분기의 역할

- `dish_flavor`: `RunnablePassthrough`가 입력 문자열을 그대로 보존한다.
- `retrieved_documents`: Retriever가 `list[Document]`를 반환한다.

### 결과 변환

1. `RunnableParallel`이 두 결과를 dict로 묶는다.
2. `format_documents()`가 각 `page_content`를 구분선으로 연결한다.
3. `build_retrieval_context()`가 Prompt 변수 이름에 맞춘 dict를 반환한다.

최종 구조는 `{'dish_flavor': str, 'wine_reviews': str}`이다.


In [14]:
from langchain_core.documents import Document
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

# 음식 이미지 -> 영어 한 문장 변환한 것 == 풍미 문자열

# 풍미 문자열을 보존하면서 검색하고,
# 생성 프롬프트용 context dict 변환

# Retriever 검색 결과(list[Document])에서
# page_content만 따로 모아둔 하나의 문자열 반환
def format_documents(documents: list[Document]) -> str:
    return '\n\n--- review ---\n\n'.join(
        document.page_content for document in documents
    )

def build_retrieval_context(payload: dict) -> dict:
    return {
        'dish_flavor': payload['dish_flavor'], # 원문
        'wine_reviews': format_documents(payload['retrieved_documents']),
        # 리뷰 5개를 하나의 문자열로 변환
    }

# 병렬처리

# 1. 하나의 풍미 문자열을 두 분기에 전달하고 결과 dict 반환
retrieve_with_query = RunnableParallel(
    dish_flavor = RunnablePassthrough(), # 풍미 문자열
    retrieved_documents=wine_retriever, # list[Documents]
)

# 2. build_retrieval_context를 RunnableLambda로 감싸기
wine_review_retrieval_chain = (
    retrieve_with_query # dict(원문, 검색 결과 list)
    | RunnableLambda(build_retrieval_context)
)
# {
#   'dish_flavor': 음식 이미지를 영어 한 문장으로 변환한 것
#   'wine_reviews': Pinecone에서 영어문장과 유사한 리뷰 5개를 하나의 문자열로 변환한 것
# }

### Retrieval Runnable의 출력 확인하기

Generation에 연결하기 전에 반환 dict의 구조를 확인한다.

- `dish_flavor`: 처음 입력한 풍미 query 문자열이다.
- `wine_reviews`: 검색된 리뷰 본문을 합친 context 문자열이다.

두 key 이름은 다음 `ChatPromptTemplate`의 `{dish_flavor}`, `{wine_reviews}`와 정확히 같아야 한다.


In [17]:
retrieval_payload = wine_review_retrieval_chain.invoke(
    sample_dish_flavor # 음식 이미지 -> 영어 한 문장 샘플
)

print('payload key:', list(retrieval_payload))

print('dish_flavor:', retrieval_payload['dish_flavor'])

print('wine_reviews:', retrieval_payload['wine_reviews'][:1000])

# dish_flavor : LLM에게 전달할 요리 설명(==검색어)
# wine_reviews: LLM에게 전달할 검색 근거
#   -> wine_reviews에 작성된 리뷰 5개를 이용해서
#      LLM이 최종 대답을 생성

payload key: ['dish_flavor', 'wine_reviews']
dish_flavor: A succulent roast beef garnished with fresh rosemary and accompanied by vibrant cherry tomatoes and roasted vegetables.
wine_reviews: : 28592
country: US
description: Cherry, pepper and warming spices lead on the red from Virginia. On the palate, cedar, spice and pepper are delivered in a big package. Pair with grilled meat, spicy cuisine.
designation: Rooster Red Wine
points: 83
price: 20.0
province: Virginia
region_1: Virginia
region_2: 
taster_name: Susan Kostrzewa
taster_twitter_handle: @suskostrzewa
title: Veramar NV Rooster Red Wine Red (Virginia)
variety: Bordeaux-style Red Blend
winery: Veramar

--- review ---

: 12969
country: US
description: Roasted and raw beef, dried strawberry, crushed marjoram and salt-and-pepper seasonings show on this Syrah. Extremely dark in the glass, it's bolstered by 4% Grenache, 4% Mourvèdre and 2% Viognier. Elderberry fruit kicks off the palate, following by pepper, coffee grinds, espresso 

### 검색 근거로 추천 Runnable 만들기

Generation은 Retrieval이 만든 dict를 Prompt에 채워 최종 추천을 생성한다.

- 입력: `{'dish_flavor': str, 'wine_reviews': str}`이다.
- Prompt: 요리 풍미와 검색 리뷰를 각각 지정된 위치에 넣는다.
- Model: 리뷰를 읽고 한국어 추천을 생성한다.
- Parser: `AIMessage`를 추천 문자열로 바꾼다.

`recommend_wines()`도 결과 문자열이 아니라 Prompt → Model → Parser Runnable을 반환한다. 뒤의 `RunnableLambda`가 반환된 Runnable을 같은 입력 dict로 이어서 실행한다.

프롬프트 지시만으로 사실 일치가 완전히 보장되지는 않는다. 추천 이름과 이유가 실제 `wine_reviews`에 있는지 별도로 확인한다.


In [19]:
def recommend_wines(query: dict):

    # 1. Prompt 작성
    recommend_wines_prompt = ChatPromptTemplate.from_messages([
            ('system', '''페르소나: 당신은 와인과 음식 페어링에 열정을 지닌 지식 많고 경험 풍부한 소믈리에이다. 다양한 와인 산지, 포도 품종과 테이스팅 노트를 폭넓게 이해한다. 친근하고 다가가기 쉬운 태도로 초보자와 애호가 모두가 와인을 편하게 접할 수 있도록 돕는다.

    역할: 소믈리에로서 다양한 요리에 잘 어울리는 와인을 전문적으로 추천한다. 이용자가 새로운 와인을 탐색하도록 안내하고 와인 테이스팅의 섬세한 차이를 이해하도록 돕는다. 적절한 와인과 요리를 연결해 식사 경험을 향상하는 것이 목표이다.

    예시:

    구운 마늘 버터 새우에 어울리는 와인을 묻는다면 Chardonnay 또는 Albariño를 제안하고, 와인의 산미가 요리의 풍부한 맛과 기름진 느낌을 어떻게 균형 있게 잡아 주는지 설명한다.
    가격이 합리적이면서 품질 좋은 와인을 묻는다면 여러 산지의 구체적인 선택지를 추천하고 각 와인의 풍미 특성과 어울리는 음식을 설명한다.
    와인 보관법을 설명할 때는 와인의 품질을 유지할 수 있는 온도, 습도와 적절한 보관 조건을 실용적으로 안내한다.'''),
            ('human', '''
    와인 페어링 추천해주세요.
    아래의 요리설명과 와인리뷰만을 참고하여 한글로 답변해주세요.
    아래의 요리설명과 와인리뷰외의 내용을 추가하지 말아주세요.

    요리설명:
    {dish_flavor}

    와인리뷰:
    {wine_reviews}

    추천 와인과 이유:
    '''),
        ])

    # 2. LLM 생성
    recommendation_llm = ChatOpenAI(
        model=CHAT_MODEL_NAME,
        use_responses_api=True,
        temperature=0,
        reasoning_effort='none'
    )

    # 3. LLM의 AIMessage를 str로 변환
    output_parser = StrOutputParser()

    # 4. 위 3개 chain == Generation
    return (
        recommend_wines_prompt
        | recommendation_llm
        | output_parser
    )


### 전체 이미지 RAG Chain 연결하기

세 Runnable을 `|`로 연결한다. 각 단계의 출력 자료형이 다음 단계의 입력 자료형과 이어져야 한다.

1. `describe_dish_flavor_chain`: `{'image_urls': list[str]}` → 풍미 `str`이다.
2. `wine_review_retrieval_chain`: 풍미 `str` → `{dish_flavor, wine_reviews}` dict이다.
3. `recommend_wines_chain`: 검색 context dict → 추천 `str`이다.

이미지 해석은 query 전처리이고, Retrieval과 Generation이 2-step RAG 본체이다. 최종 Chain은 추천 문자열만 반환하므로 검색 근거는 앞의 Retrieval 출력 확인 단계에서 별도로 검토한다.


In [22]:
# 위에서 만든 모든 chain을 하나로 연결

# 1. Generation 함수를 RunnableLambda로 감싸기
recommend_wines_chain = RunnableLambda(recommend_wines)

# 2. 모든 chain 연결
sommelier_rag_chain = (

    # 음식 이미지-> 풍미 문자열
    describe_dish_flavor_chain

    # 풍미 문자열과 유사한 리뷰를 Pinecon에서 조회
    | wine_review_retrieval_chain

    # 풍미 문자열 + 유사 리뷰를 이용해서 LLM에서 답변 생성 지시
    | recommend_wines_chain
)

# 호출/결과 확인
# recommendation = sommelier_rag_chain.invoke({
#     'image_urls': [
#         'https://static.wtable.co.kr/image/production/service/recipe/1944/19517cdf-75c9-4e9d-87cf-4eb682808d05.jpg?size=800x800'
#         # 'https://recipe1.ezmember.co.kr/cache/recipe/2016/06/10/40fd7656a0a3e0c25735138237b806ee1.jpg'
#     ]
# })

# print(recommendation)

# stream 버전
rag_input = {
    'image_urls': [
        'https://static.wtable.co.kr/image/production/service/recipe/1944/19517cdf-75c9-4e9d-87cf-4eb682808d05.jpg?size=800x800'
    ]
}

recommendation_chunks = []

for chunk in sommelier_rag_chain.stream(rag_input):
    print(chunk, end='', flush=True)
    recommendation_chunks.append(chunk)

# 재사용 대비
recommendation = ''.join(recommendation_chunks)

### 추천 와인과 이유

1. **Pace 2013 Syrah (Santa Ynez Valley) — 93점, $35**
   - 가장 잘 어울리는 선택입니다.
   - 말린 소고기, 간장, 숯불, 검은 후추, 검은깨 같은 풍미가 요리의 소고기, 간장, 마늘, 참깨와 직접적으로 연결됩니다.
   - 와인의 강한 감칠맛과 후추 풍미가 부드럽고 육즙 많은 갈비와 잘 어울립니다.

2. **Eberle 2015 Steinbeck Vineyard Syrah (Paso Robles) — 91점, $28**
   - 간장 풍미, 데리야키 글레이즈 스테이크, 숯, 훈연 향이 특징이라 요리의 간장 양념과 구수한 감칠맛에 잘 맞습니다.
   - 검은 자두와 오디 풍미가 요리의 은은한 단맛을 보완하고, 훈연 향이 풍부한 갈비와 조화를 이룹니다.

3. **Pomar Junction 2012 Syrah (Paso Robles) — 89점, $38**
   - 간장, 소고기 숯불구이, 통후추, 검은깨 풍미가 요리의 핵심 재료와 잘 맞습니다.
   - 생기 있는 산미가 진하고 기름진 갈비의 느낌을 정돈하며, 후추와 밝은 베리 풍미가 고추의 은은한 매운맛과 어울립니다.

4. **Hey Mambo 2013 Sultry Red (California) — 85점, $12**
   - 살짝 달콤하고 오크, 후추, 스모키한 풍미가 있어 양념이 진한 갈비와 잘 맞습니다.
   - 소고기와 액체 연기 같은 풍미가 요리의 진한 감칠맛과 조화를 이루며, 합리적인 가격이 장점입니다.

5. **kukkula 2012 Sisu Red (Paso Robles) — 93점, $45**
   - 오디, 제비꽃, 후추, 흙, 코코아, 간장 풍미가 어우러져 요리의 감칠맛과 향신료 풍미에 잘 연결됩니다.
   - 다만 다른 와인보다 과실과 꽃 향이 두드러져, 진한 간장 풍미와 고추 향을 부드럽게 감싸는 선택에 가깝습니다.

**최종 추천:** **Pace 2013 Syrah**  
간장, 소고기, 숯불, 후추,